# Notebook 2 — Success scoring (TextGrid transcripts)

Uses the human-annotated transcripts pulled from the Balance Corpus API
(`fetch_textgrid_transcripts.py`).

Two success measures, reconciled into one adjudicated result:

1. **Duration-based (crude)** — the game stops the instant the guesser gets it right. The
   60s game clock starts when the clue-giver *starts speaking*, not at the recording's t=0, so
   that offset is subtracted before applying the threshold.
2. **Text-based** — did the *guesser* say the target word **before time ran out**? This
   isn't just "does the word appear anywhere in the transcript" — a naive version of that
   check produces false positives/negatives for several real reasons found in this corpus:
   - **Spelling variants** ("donut" vs "doughnut")
   - **Irregular plurals** ("puppies" vs "puppy" — not just a trailing 's')
   - **Segments split across two intervals** by the transcriber/annotator ("zip" + "per" for
     "zipper")
   - **The clue-giver accidentally saying the target word** — the recording can still stop
     early in this case (duration-based method alone would misread it as a successful guess),
     but the guesser didn't actually guess anything, so it isn't a real success
   - **The word said after the 60s timer expired** — turns up in the transcript, but too
     late to count



## 0. Load the data

In [2]:
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_textgrid_dir(dirname="textgrid_transcripts", start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        cand = os.path.join(d, dirname)
        if os.path.isdir(cand) and os.path.isfile(os.path.join(cand, "transcripts_all.csv")):
            return cand
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    raise FileNotFoundError(f"Could not find a '{dirname}' folder with transcripts_all.csv nearby.")

TEXTGRID_DIR = find_textgrid_dir()
print("Using:", TEXTGRID_DIR)

combined = pd.read_csv(os.path.join(TEXTGRID_DIR, "transcripts_all.csv"))
dur_df = pd.read_csv(os.path.join(TEXTGRID_DIR, "trial_durations.csv"))
print(f"{len(combined)} segments across {combined.trial_id.nunique()} trials")


Using: /Volumes/TimMurphy/wobblyFriendships/Transcript Analysis/textgrid_transcripts
5012 segments across 400 trials


## 1. Word matcher

Matches whole words only (fixed a real bug in the earlier version: a trailing `\w*` after the
word boundary let "cat" match inside "catalogue", since `\b` only anchors the *start* and the
greedy `\w*` consumed the rest up to the next boundary — which is just the end of the word.
Removed that; matches are now built from an explicit set of acceptable forms instead.

The option is there to add more words to `TARGET_WORD_ALIASES` if more spelling variants are spotted. I have not done an exhaustive check of this due to time constraints.


In [ ]:
TARGET_WORD_ALIASES = {
    "doughnut": ["donut"],
}

MAX_SPLIT_GAP = 1.0   # seconds

def normalize_alpha(s):
    return re.sub(r"[^a-z]", "", str(s).lower())

def morphological_variants(word):
    """Regular plural/inflection forms. Doesn't attempt full lemmatisation -- just the common
    English patterns (regular 's'/'es', and the 'y' -> 'ies' irregular)."""
    w = word.lower()
    v = {w, w + "s", w + "es"}
    if w.endswith("y") and len(w) > 1 and w[-2] not in "aeiou":
        v.add(w[:-1] + "ies")
    return v

def all_variants(word):
    words = {word.lower()} | set(TARGET_WORD_ALIASES.get(word.lower(), []))
    variants = set()
    for w in words:
        variants |= morphological_variants(w)
    return variants

def word_pattern(word):
    variants = sorted(all_variants(word), key=len, reverse=True)
    return re.compile(r"\b(?:" + "|".join(re.escape(v) for v in variants) + r")\b")

def find_word_matches(segs, target, max_gap=MAX_SPLIT_GAP):
    """segs: DataFrame of one role's segments for one trial. Returns sorted list of match
    start times, covering both single-segment matches and adjacent-pair splits."""
    if len(segs) == 0:
        return []
    pattern = word_pattern(target)
    target_norms = {normalize_alpha(v) for v in all_variants(target)}
    matches = []
    segs = segs.sort_values("start").reset_index(drop=True)
    for _, row in segs.iterrows():
        if pattern.search(str(row.text).lower()):
            matches.append(row.start)
    for i in range(len(segs) - 1):
        gap = segs.loc[i + 1, "start"] - segs.loc[i, "end"]
        if gap <= max_gap:
            concat = normalize_alpha(segs.loc[i, "text"]) + normalize_alpha(segs.loc[i + 1, "text"])
            if concat in target_norms:
                matches.append(segs.loc[i, "start"])
    return sorted(set(matches))


## 2. Game-clock start

Same as before: the clue-giver's first spoken segment marks when gameplay actually starts.


In [ ]:
cg_start = (combined[combined.role == "clue_giver"]
            .sort_values("start")
            .groupby("trial_id")["start"].first()
            .rename("clue_giver_start"))
print(cg_start.describe())


## 3. Adjudicate each trial

For every trial: find every time the target word is said, by *either* role, relative to the
clue-giver's start. A trial only counts as a genuine success if the **guesser** said it
**before the timer expired**. If the clue-giver said it (a slip) and the guesser never said it
themselves, that's flagged as `clue_giver_leak_no_guesser_success`, not a success — even if
the recording happened to stop early. If the guesser did say it but after the threshold,
that's `said_after_timer`, also not a success.

`success_duration` is still computed independently as a cross-check, but `success` (the
adjudicated column) is what should be used for analysis from here on.


In [ ]:
SUCCESS_DURATION_THRESHOLD = 59.5   # seconds of elapsed gameplay

dur_df = dur_df.merge(cg_start, on="trial_id", how="left")
dur_df["clue_giver_start"] = dur_df["clue_giver_start"].fillna(0.0)
dur_df["elapsed_duration"] = dur_df["duration"] - dur_df["clue_giver_start"]
dur_df["success_duration"] = dur_df["elapsed_duration"] < SUCCESS_DURATION_THRESHOLD

rows = []
for trial_id, trial_df in combined.groupby("trial_id"):
    target = trial_df.target_word.iloc[0]
    start_ref = cg_start.get(trial_id, 0.0)

    cg_segs = trial_df[trial_df.role == "clue_giver"]
    g_segs = trial_df[trial_df.role == "guesser"]

    cg_matches = [t - start_ref for t in find_word_matches(cg_segs, target)]
    g_matches = [t - start_ref for t in find_word_matches(g_segs, target)]

    clue_giver_leak = len(cg_matches) > 0
    clue_giver_leak_time = min(cg_matches) if clue_giver_leak else np.nan

    success_text_raw = len(g_matches) > 0
    time_to_success = min(g_matches) if success_text_raw else np.nan

    success = bool(success_text_raw and time_to_success <= SUCCESS_DURATION_THRESHOLD)

    if success:
        note = ""
    elif clue_giver_leak and not success_text_raw:
        note = "clue_giver_leak_no_guesser_success"
    elif success_text_raw and time_to_success > SUCCESS_DURATION_THRESHOLD:
        note = "said_after_timer"
    else:
        note = ""

    rows.append(dict(trial_id=trial_id, success_text_raw=success_text_raw,
                      time_to_success=time_to_success, clue_giver_leak=clue_giver_leak,
                      clue_giver_leak_time=clue_giver_leak_time, success=success, note=note))

adjudicated = pd.DataFrame(rows)
print(f"Adjudicated success rate: {adjudicated.success.mean():.1%} "
      f"({int(adjudicated.success.sum())}/{len(adjudicated)} trials)")
print("\nNote breakdown (non-successes only):")
print(adjudicated[~adjudicated.success].note.replace("", "(other/no word said at all)").value_counts())


## 4. Merge into one results table

In [ ]:
meta_cols = ["trial_id", "pair_id", "trial_number", "condition", "target_word"]
meta = combined.drop_duplicates("trial_id")[meta_cols]

results = (meta
    .merge(dur_df[["trial_id", "duration", "clue_giver_start", "elapsed_duration", "success_duration"]],
           on="trial_id", how="left")
    .merge(adjudicated, on="trial_id", how="left"))

out_path = os.path.join(TEXTGRID_DIR, "success.csv")
results.to_csv(out_path, index=False)
print(f"Saved {len(results)} trials -> {out_path}")
results.head()


## 5. Compare against duration-based (residual check)

Should now agree almost completely — the adjudicated `success` column already accounts for
the timing cutoff and clue-giver leaks, so remaining disagreements are worth an actual manual
listen rather than a matching-logic fix.


In [ ]:
agree = results.success_duration == results.success
print(f"Agreement: {agree.mean():.1%} ({int(agree.sum())}/{len(results)} trials)")
print("\nConfusion matrix (rows=duration-based, cols=adjudicated):")
print(pd.crosstab(results.success_duration, results.success))

disagree = results[~agree].sort_values("elapsed_duration")
if len(disagree):
    print(f"\n{len(disagree)} trial(s) still disagree:\n")
    print(disagree[["trial_id", "elapsed_duration", "success_duration", "success", "note"]]
          .to_string(index=False))


## 6. Figures overview

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
by_cond = results.groupby("condition")[["success_duration", "success"]].mean()
by_cond.plot.bar(ax=ax, color=["#c44", "#48a"])
ax.set_ylabel("success rate"); ax.set_ylim(0, 1)
ax.set_title("Success rate by condition"); ax.legend(["duration-based", "adjudicated"])
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
results.groupby("pair_id").success.mean().sort_values().plot.bar(ax=ax, color="#4a8")
ax.set_ylabel("success rate"); ax.set_ylim(0, 1)
ax.set_title("Success rate by pair")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
results[results.success].time_to_success.plot.hist(bins=25, ax=ax, color="#a84")
ax.set_xlabel("time to correct guess, from clue-giver start (s)"); ax.set_ylabel("count")
ax.set_title("Time to success (successful trials only)")
plt.tight_layout(); plt.show()


In [ ]:
sub = results[results.success].dropna(subset=["elapsed_duration", "time_to_success"])

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(sub.time_to_success, sub.elapsed_duration, alpha=0.6, color="#48a")
lims = [0, max(sub.elapsed_duration.max(), sub.time_to_success.max()) + 2]
ax.plot(lims, lims, "r--", label="y = x")
ax.set_xlabel("time to correct guess (s)"); ax.set_ylabel("elapsed trial duration (s)")
ax.set_title("Recording stop time vs. moment of correct guess")
ax.legend(); plt.tight_layout(); plt.show()


## 7. Hardest / easiest target words

Success **rate** and, among successes, **time taken** — a word can be high on one and low
on the other.


In [ ]:
word_stats = results.groupby("target_word").agg(
    n=("success", "size"),
    success_rate=("success", "mean"),
    mean_time_to_success=("time_to_success", "mean"),
).sort_values("success_rate")

print(word_stats.to_string())

fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(word_stats))))
word_stats.success_rate.plot.barh(ax=ax, color="#c66")
ax.set_xlabel("success rate"); ax.set_xlim(0, 1)
ax.set_title("Success rate by target word (hardest at top)")
plt.tight_layout(); plt.show()


In [ ]:
time_ranked = word_stats.dropna(subset=["mean_time_to_success"]).sort_values("mean_time_to_success", ascending=False)

fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(time_ranked))))
time_ranked.mean_time_to_success.plot.barh(ax=ax, color="#a84")
ax.set_xlabel("mean time to correct guess (s)")
ax.set_title("Average time to success by target word (slowest at top, successful trials only)")
plt.tight_layout(); plt.show()


## 8. Difficulty by condition

In [ ]:
pivot = results.pivot_table(index="target_word", columns="condition", values="success", aggfunc="mean")
pivot_n = results.pivot_table(index="target_word", columns="condition", values="success", aggfunc="size")

print("Success rate by word x condition:")
print(pivot.round(2).to_string())
print("\nN per cell:")
print(pivot_n.to_string())

fig, ax = plt.subplots(figsize=(6, max(3, 0.35 * len(pivot))))
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Success rate: target word x condition")
fig.colorbar(im, ax=ax, label="success rate")
plt.tight_layout(); plt.show()
